# PS S5E11 - Dataset Profiling (DataFlow v3)

*Auto-generated by DataFlow*

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Discover available input directories
KAGGLE_INPUT = '/kaggle/input'
if os.path.exists(KAGGLE_INPUT):
    print('Available input directories:')
    for d in os.listdir(KAGGLE_INPUT):
        print(f'  {d}')
else:
    print('Not running on Kaggle, checking local paths')

In [ ]:
INPUT_DIR = "/kaggle/input/playground-series-s5e11"

# Search for data in multiple possible locations
candidates = [
    INPUT_DIR,
    "/kaggle/input/competitions/playground-series-s5e11",
]

found = False
for candidate in candidates:
    if os.path.exists(candidate) and os.listdir(candidate):
        INPUT_DIR = candidate
        found = True
        break

if not found:
    # Deep search: walk /kaggle/input/ recursively for CSV files
    for root, dirs, files in os.walk("/kaggle/input"):
        csvs = [f for f in files if f.endswith(".csv")]
        if csvs:
            INPUT_DIR = root
            found = True
            break

print(f"Using: {INPUT_DIR}")
assert os.path.exists(INPUT_DIR), f"Data directory not found: {INPUT_DIR}"
print("Files:")
for f in sorted(os.listdir(INPUT_DIR)):
    fpath = os.path.join(INPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Load train and test sets
csv_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')]
dataframes = {}
for f in csv_files:
    name = f.replace('.csv', '')
    dataframes[name] = pd.read_csv(os.path.join(INPUT_DIR, f))
    print(f'{name}: {dataframes[name].shape}')

# Use 'train' as primary if it exists, else first file
if 'train' in dataframes:
    df = dataframes['train']
    primary_name = 'train'
else:
    primary_name = list(dataframes.keys())[0]
    df = dataframes[primary_name]
print(f'\nPrimary dataset: {primary_name} {df.shape}')

## Schema Analysis

In [ ]:
print('=== SCHEMA ===')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print()
print(df.dtypes.to_string())

## Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
if len(missing_df) > 0:
    print('=== MISSING VALUES ===')
    print(missing_df.to_string())
else:
    print('No missing values found.')

## Statistical Summary

In [ ]:
print('=== NUMERIC FEATURES ===')
print(df.describe().T.to_string())

In [ ]:
print('=== CATEGORICAL FEATURES ===')
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    print(f'\n--- {col} ---')
    vc = df[col].value_counts()
    print(f'  Unique: {df[col].nunique()}')
    print(f'  Top values:')
    print(vc.head(10).to_string())

## Cardinality Analysis

In [ ]:
cardinality = df.nunique().sort_values(ascending=False)
print('=== CARDINALITY ===')
print(cardinality.to_string())

## Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
if numeric_df.shape[1] > 1:
    corr = numeric_df.corr()
    print('=== TOP CORRELATIONS ===')
    # Get upper triangle of correlation matrix
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.unstack().dropna().sort_values(key=abs, ascending=False)
    print(pairs.head(20).to_string())
else:
    print('Not enough numeric features for correlation analysis.')

## Sample Rows

In [ ]:
df.head(10)

## Structured Output

In [ ]:
profile = {
    'shape': list(df.shape),
    'columns': list(df.columns),
    'dtypes': {col: str(dtype) for col, dtype in df.dtypes.items()},
    'missing': {col: int(cnt) for col, cnt in df.isnull().sum().items() if cnt > 0},
    'missing_pct': {col: round(cnt / len(df) * 100, 2) for col, cnt in df.isnull().sum().items() if cnt > 0},
    'cardinality': {col: int(n) for col, n in df.nunique().items()},
    'numeric_stats': json.loads(df.describe().T.to_json()) if len(df.select_dtypes(include=[np.number]).columns) > 0 else {},
}
print('===PROFILE_JSON_START===')
print(json.dumps(profile, indent=2))
print('===PROFILE_JSON_END===')